# Leuven Bike Road Dataset — Inference Features

Builds a dataset matching `sites_enriched_clean.csv` schema for all bike-accessible
road segments within the administrative boundary of Leuven.

**Output:** `explo/andy/output/leuven_bike_roads.csv`

## 1  Imports & Settings

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
warnings.filterwarnings('ignore')
ox.settings.log_console = False

OUTPUT_DIR  = 'explo/andy/output'
CACHE_DIR   = 'explo/andy/output/leuven_cache'
OUTPUT_PATH = f'{OUTPUT_DIR}/leuven_bike_roads.csv'
os.makedirs(CACHE_DIR, exist_ok=True)

POI_RADII = [250, 500, 1000]
POI_TAGS  = {
    'shop'      : {'shop': True},
    'education' : {'amenity': ['school', 'university', 'college', 'kindergarten']},
    'hotel'     : {'tourism': ['hotel', 'hostel', 'motel', 'guest_house']},
    'hospital'  : {'amenity': ['hospital', 'clinic', 'doctors', 'pharmacy']},
}

# ── Feature mappings: OSM highway type → training data vocabulary ─────────────
HIGHWAY_TO_MORPHOLOGY = {
    'cycleway'      : 'cycleway_footpath',
    'footway'       : 'cycleway_footpath',
    'path'          : 'cycleway_footpath',
    'track'         : 'cycleway_footpath',
    'pedestrian'    : 'cycleway_footpath',
    'bridleway'     : 'cycleway_footpath',
    'residential'   : 'single_carriageway',
    'living_street' : 'single_carriageway',
    'unclassified'  : 'single_carriageway',
    'tertiary'      : 'single_carriageway',
    'tertiary_link' : 'single_carriageway',
    'secondary'     : 'divided_road_non_motorway',
    'secondary_link': 'divided_road_non_motorway',
    'primary'       : 'divided_road_non_motorway',
    'primary_link'  : 'divided_road_non_motorway',
    'trunk'         : 'divided_road_non_motorway',
    'service'       : 'service_road',
}
HIGHWAY_TO_CAT_CODE = {
    'cycleway'      : '-9',  'footway'       : '-9',
    'path'          : '-9',  'track'         : '-9',
    'pedestrian'    : '-9',  'bridleway'     : '-9',
    'living_street' : 'EW',  'service'       : 'EW',  'residential'   : 'EW',
    'unclassified'  : 'IW',  'tertiary'      : 'IW',  'tertiary_link' : 'IW',
    'secondary'     : 'OW',  'secondary_link': 'OW',
    'primary'       : 'RW',  'primary_link'  : 'RW',
    'trunk'         : 'VHW',
}
CAT_CODE_TO_EN = {
    'EW': 'local_access_road', 'IW': 'local_road',
    'OW': 'minor_road',        'RW': 'regional_road',
    'VHW': 'high_capacity_road', '-9': 'not_applicable',
}
print('Settings loaded.')

## 2  Fetch Leuven Boundary & Bike Road Network

In [ ]:
GRAPH_CACHE = f'{CACHE_DIR}/leuven_bike_graph.graphml'

if os.path.exists(GRAPH_CACHE):
    G = ox.load_graphml(GRAPH_CACHE)
    print(f'Loaded bike network from cache.')
else:
    print('Fetching Leuven bike road network from OSM...')
    t0 = time.time()
    G = ox.graph_from_place('Leuven, Belgium', network_type='bike', retain_all=False)
    ox.save_graphml(G, GRAPH_CACHE)
    print(f'Done in {time.time()-t0:.1f} s — cached.')

print(f'Nodes: {len(G.nodes):,}   Edges (segments): {len(G.edges):,}')

## 3  Convert Graph Edges to GeoDataFrame

In [ ]:
# Get edges (road segments) as GeoDataFrame
edges = ox.graph_to_gdfs(G, nodes=False).reset_index()
print(f'Total segments : {len(edges):,}')
print(f'CRS            : {edges.crs}')

# Compute segment midpoint (centroid) in WGS84
edges['longitude'] = edges.geometry.centroid.x
edges['latitude']  = edges.geometry.centroid.y

# Compute segment length in metres (project to Lambert 72)
edges_lam = edges.to_crs('EPSG:31370')
edges['length_m'] = edges_lam.geometry.length.round(2)

# Normalise highway type (OSM sometimes stores as list)
edges['highway_type'] = edges['highway'].apply(
    lambda h: h[0] if isinstance(h, list) else h
)

print('\nhighway_type distribution:')
print(edges['highway_type'].value_counts().head(15))

## 4  Map OSM Attributes → Training Schema Features

In [ ]:
# ── Road name ─────────────────────────────────────────────────────────────────
edges['road_name'] = edges.get('name', pd.Series('Unknown', index=edges.index))
if isinstance(edges['road_name'].iloc[0], list):
    edges['road_name'] = edges['road_name'].apply(
        lambda x: x[0] if isinstance(x, list) else x
    )
edges['road_name'] = edges['road_name'].fillna('Unknown')

# ── Morphology & road category ────────────────────────────────────────────────
edges['morphology_en']       = edges['highway_type'].map(HIGHWAY_TO_MORPHOLOGY).fillna('single_carriageway')
edges['road_category_code']  = edges['highway_type'].map(HIGHWAY_TO_CAT_CODE).fillna('IW')
edges['road_category_en']    = edges['road_category_code'].map(CAT_CODE_TO_EN)

# ── Bike lane width ───────────────────────────────────────────────────────────
WIDTH_TAGS = ['cycleway:width', 'cycleway:right:width', 'cycleway:left:width', 'cycleway:both:width', 'width']

def _parse_width(val):
    if pd.isna(val): return np.nan
    s = str(val).lower().strip()
    if s.endswith('cm'):
        try: return float(s.replace('cm','').strip()) / 100
        except: pass
    for u in [' meters',' meter',' m','m']: s = s.replace(u,'').strip()
    try: return float(s)
    except: return np.nan

def get_width_from_row(row):
    for tag in WIDTH_TAGS:
        if tag in row.index and not pd.isna(row.get(tag)):
            w = _parse_width(row[tag])
            if not np.isnan(w) and 0 < w < 20:
                return w
    return np.nan

edges['bike_lane_width_m'] = edges.apply(get_width_from_row, axis=1)

# Impute missing with median per morphology type
overall_med = edges['bike_lane_width_m'].median()
edges['bike_lane_width_m'] = edges.groupby('morphology_en')['bike_lane_width_m'].transform(
    lambda x: x.fillna(x.median() if not np.isnan(x.median()) else overall_med)
)
edges['bike_width_imputed'] = edges['bike_lane_width_m'].isna()
edges['bike_lane_width_m']  = edges['bike_lane_width_m'].fillna(overall_med)

# ── Other static features ─────────────────────────────────────────────────────
edges['has_cycleway']       = True   # all segments are bike-accessible by definition
edges['bike_lane_source']   = 'osm_bike_network'
edges['access_en']          = 'public_road'
edges['geometry_method_en'] = 'surveyed'
edges['municipality']       = 'Leuven'
edges['district_code']      = 'AWV214'
edges['road_code']          = edges['osmid'].astype(str)  # use OSM way ID
edges['operator']           = 'Unknown'
edges['interval_min']       = 60
edges['install_date']       = pd.NaT
edges['sensor_age_days']    = 0
edges['install_year']       = pd.Timestamp.today().year
edges['manager_code']       = '24062'        # NIS code for Leuven
edges['manager_label']      = 'Stad Leuven'
edges['dist_to_segment_m']  = 0.0
edges['segment_id']         = edges['osmid']
edges['site_id']            = edges['osmid']
edges['site_name']          = edges['road_name']

print(f'Features mapped. bike_lane_width_m imputed: {edges["bike_width_imputed"].sum()} segments')
print('\nMorphology distribution:')
print(edges['morphology_en'].value_counts())

## 5  POI Counts (fetch once per category, count locally)

In [ ]:
# Project edges to Lambert 72 for accurate metre-based buffers
edges_lam = edges.set_geometry(
    gpd.GeoSeries(
        [Point(lon, lat) for lon, lat in zip(edges['longitude'], edges['latitude'])],
        crs='EPSG:4326'
    )
).to_crs('EPSG:31370')

# Leuven bounding box (north, south, east, west) for OSMnx 2.x
LEUVEN_BBOX = (50.92, 50.83, 4.73, 4.62)

poi_gdfs = {}
print('Downloading POI data for Leuven (1 request per category, cached)...')
for cat, tags in POI_TAGS.items():
    cache_path = f'{CACHE_DIR}/poi_{cat}.gpkg'
    if os.path.exists(cache_path):
        poi_gdfs[cat] = gpd.read_file(cache_path)
        print(f'  {cat:12s}: {len(poi_gdfs[cat]):,} features from cache')
    else:
        t0 = time.time()
        # Disable OSMnx cache to avoid stale responses
        _was = ox.settings.use_cache
        ox.settings.use_cache = False
        try:
            gdf = ox.features_from_bbox(bbox=LEUVEN_BBOX, tags=tags)
            gdf = gdf.copy()
            gdf['geometry'] = gdf.geometry.centroid
            gdf = gdf.to_crs('EPSG:31370')
            gdf[['geometry']].to_file(cache_path, driver='GPKG')
            poi_gdfs[cat] = gdf
            print(f'  {cat:12s}: {len(gdf):,} features in {time.time()-t0:.1f}s — cached')
        except Exception as e:
            print(f'  {cat:12s}: no features ({type(e).__name__}) — filling 0')
            poi_gdfs[cat] = None
        finally:
            ox.settings.use_cache = _was

print('\nCounting POIs per segment per radius (local — no API calls)...')
t0 = time.time()
for cat, poi_gdf in poi_gdfs.items():
    for r in POI_RADII:
        col = f'poi_{cat}_{r}m'
        if poi_gdf is None or poi_gdf.empty:
            edges[col] = 0
        else:
            poi_geom = poi_gdf.geometry
            edges[col] = [
                int(poi_geom.within(pt.buffer(r)).sum())
                for pt in edges_lam.geometry
            ]
print(f'Done in {time.time()-t0:.1f} s')

## 6  Assemble Final Dataset — Match Training Schema

In [ ]:
# Columns in exactly the same order as sites_enriched_clean.csv
TRAINING_COLS = [
    'site_id', 'site_name', 'longitude', 'latitude',
    'municipality', 'district_code', 'road_code', 'operator',
    'interval_min', 'install_date', 'segment_id', 'road_name',
    'morphology_en', 'road_category_code', 'road_category_en',
    'access_en', 'geometry_method_en', 'manager_code', 'manager_label',
    'length_m', 'dist_to_segment_m', 'bike_lane_width_m',
    'has_cycleway', 'bike_lane_source',
    'poi_shop_250m', 'poi_shop_500m', 'poi_shop_1000m',
    'poi_education_250m', 'poi_education_500m', 'poi_education_1000m',
    'poi_hotel_250m', 'poi_hotel_500m', 'poi_hotel_1000m',
    'poi_hospital_250m', 'poi_hospital_500m', 'poi_hospital_1000m',
    'sensor_age_days', 'install_year', 'bike_width_imputed',
]

leuven_df = edges[TRAINING_COLS].copy()

# Remove duplicate segments (parallel edges between same nodes)
leuven_df = leuven_df.drop_duplicates(subset='segment_id').reset_index(drop=True)

print(f'Final dataset: {leuven_df.shape[0]:,} segments × {leuven_df.shape[1]} columns')
print(f'Missing values: {leuven_df.isna().sum().sum()}')
display(leuven_df.head(5))

## 7  Validate Against Training Schema

In [ ]:
import matplotlib.pyplot as plt

train_df = pd.read_csv('explo/andy/output/sites_enriched_clean.csv')

# ── Column match check ────────────────────────────────────────────────────────
missing_cols = set(train_df.columns) - set(leuven_df.columns)
extra_cols   = set(leuven_df.columns) - set(train_df.columns)
print('Columns in training but missing in Leuven :', missing_cols or 'None ✅')
print('Extra columns in Leuven not in training    :', extra_cols   or 'None ✅')

# ── Compare key numeric distributions ─────────────────────────────────────────
compare_cols = ['bike_lane_width_m', 'length_m',
                'poi_shop_1000m', 'poi_education_1000m']
fig, axes = plt.subplots(2, len(compare_cols), figsize=(16, 7), sharey=False)
for j, col in enumerate(compare_cols):
    for i, (label, data) in enumerate([('Training (sites)', train_df), ('Leuven roads', leuven_df)]):
        ax = axes[i][j]
        ax.hist(data[col].dropna(), bins=25, color=['#3498db','#2ecc71'][i],
                edgecolor='white', alpha=0.85)
        ax.set_title(f'{col}\n{label}', fontsize=10)
        ax.set_ylabel('Count')
plt.suptitle('Feature Distribution: Training vs Leuven Inference Dataset', fontsize=13)
plt.tight_layout()
plt.show()

# ── Morphology comparison ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, data) in zip(axes, [('Training', train_df), ('Leuven', leuven_df)]):
    counts = data['morphology_en'].value_counts()
    ax.barh(counts.index, counts.values, color='#9b59b6', edgecolor='white')
    ax.set_title(f'morphology_en — {label}', fontsize=11)
    ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 8  Export

In [ ]:
leuven_df.to_csv(OUTPUT_PATH, index=False)
print(f'✅  Saved: {OUTPUT_PATH}')
print(f'   {leuven_df.shape[0]:,} road segments × {leuven_df.shape[1]} columns')
print()
print('Summary:')
display(pd.DataFrame({
    'dtype'  : leuven_df.dtypes,
    'missing': leuven_df.isna().sum(),
    'unique' : leuven_df.nunique(),
}))